<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/session05-discussion-3.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Session 5: in-class discussion problem (3 of 3)

This is the block 3 question of the lecture quiz. First guess the answer in the quiz, on your own.
Then run the code below and look at what it does. There is one mistake in it, but finding that is
only the way in: the point is to work out why the result comes out as it does. Discuss that with
your neighbour, then answer the quiz again.

## One protein, three domains: which match is the most significant?

**BRCA1** (UniProt `P38398`, the breast-cancer susceptibility gene) has three Pfam domain matches:

- a RING finger (Pfam PF00097), 41 residues long;
- a BRCT domain (PF00533), the domain the BRCT family is named after, 79 residues;
- a serine-rich domain found next to BRCT domains (PF12820), 164 residues.

All three domains are real. Which match has the smallest E-value, that is, the most significant
match?

**Fix and run.** The cell below lists the three matches, most significant first, but the line marked
`FIX` sorts them the wrong way round. Run it once as it is: which domain comes out on top? Then fix
the line and run it again.

In [1]:
# BRCA1's Pfam matches, as the InterPro API returned them in October 2026
# (the cell after "If you finish early" fetches them live).
hits = [
    {"pfam": "PF00097", "name": "Zinc finger, C3HC4 type (RING finger)", "start": 24, "end": 64, "evalue": 1.6e-06},
    {"pfam": "PF00533", "name": "BRCA1 C Terminus (BRCT) domain", "start": 1645, "end": 1723, "evalue": 4.6e-07},
    {"pfam": "PF12820", "name": "Serine-rich domain associated with BRCT", "start": 345, "end": 508, "evalue": 2.8e-77},
]

ranked = sorted(hits, key=lambda h: h["evalue"], reverse=True)   # FIX: most significant first

print(f"{'rank':<6}{'Pfam':<10}{'length':>7}{'E-value':>10}   name")
for rank, h in enumerate(ranked, 1):
    print(f"{rank:<6}{h['pfam']:<10}{h['end'] - h['start'] + 1:>7}{h['evalue']:>10.1e}   {h['name']}")

rank  Pfam       length   E-value   name
1     PF00097        41   1.6e-06   Zinc finger, C3HC4 type (RING finger)
2     PF00533        79   4.6e-07   BRCA1 C Terminus (BRCT) domain
3     PF12820       164   2.8e-77   Serine-rich domain associated with BRCT


**If you finish early.**

1. BLAST and HMMER also report a bit score. Would you sort by bit score in the same direction as by
   E-value?
2. Run the next cell to fetch the matches live from InterPro (it needs the internet). Are they the
   same?

In [2]:
import requests

r = requests.get("https://www.ebi.ac.uk/interpro/api/entry/pfam/protein/uniprot/P38398/", timeout=10)
r.raise_for_status()
for entry in r.json()["results"]:
    loc = entry["proteins"][0]["entry_protein_locations"][0]    # "score" here is the domain E-value
    frag = loc["fragments"][0]
    print(f"{entry['metadata']['accession']:<10}{frag['start']:>5}-{frag['end']:<6}{loc['score']:>10}   "
          f"{entry['metadata']['name']}")

PF00097      24-64       1.6e-06   Zinc finger, C3HC4 type (RING finger)
PF00533    1645-1723     4.6e-07   BRCA1 C Terminus (BRCT) domain
PF12820     345-508      2.8e-77   Serine-rich domain associated with BRCT


<details>
<summary><b>Discussion point</b> &mdash; open this once you have run the code and talked it through</summary>

As it was, the cell sorted the largest E-value first, as if a bigger
number meant a better match, and put the RING finger on top. For an E-value it is the other way
round: it is the number of matches this good expected by chance, so *smaller* is more significant.
(For a bit score, bigger is better.) Fixed, the serine-rich domain comes first, with
$E = 2.8 \times 10^{-77}$, then BRCT ($4.6 \times 10^{-7}$) and the RING finger
($1.6 \times 10^{-6}$).

An E-value measures how surprising the match is, not how real or important the domain is. The
serine-rich domain is long, so the score adds up over 164 positions, and its family consists of
closely related BRCA1 orthologues, so the profile is very specific. The RING finger is short and its
family very diverse: apart from the zinc-binding cysteines and histidines, its members share
little sequence. A weaker but still significant E-value is weaker evidence from sequence alone, not
a sign that the domain is absent; other evidence, such as a structure or the conserved zinc-binding
residues, can strengthen it.

</details>